In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import pickle
import os, sys
import torch
from matplotlib.lines import Line2D
from collections import defaultdict
from matplotlib.colors import LinearSegmentedColormap
from tqdm.autonotebook import tqdm
# Root project
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))

if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root added to sys.path:", project_root)

from envs import *
from basal_ganglia import *
from stn_gpe import *

# filter warnings
import warnings
warnings.filterwarnings("ignore")



plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],  # Times for publication
    "font.size": 8,                    # Base font size
    "axes.labelsize": 8,              # Axis label font size
    "axes.labelweight": "bold",
    "font.weight": "bold",   
    "axes.titlesize":8,              # Axis title (not fig title)
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize":8,              # Legend font size

    "figure.dpi": 300,                 # Display resolution
    "savefig.dpi": 300,                # Save resolution
    "figure.constrained_layout.use": True,  # Better spacing than tight_layout

    "axes.linewidth": 0.8,             # Thin but clear axes lines
    "lines.linewidth": 1.2,
    "lines.markersize": 5,

    "axes.spines.top": False,
    "axes.spines.right": False,

    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 4,
    "ytick.major.size": 4,
    "xtick.minor.size": 2,
    "ytick.minor.size": 2,

    "legend.frameon": False,
    "legend.handlelength": 2.0,        # Slightly longer legend lines
    "legend.handletextpad": 0.5,

    "pdf.fonttype": 42,                # Embed editable fonts in PDF
    "ps.fonttype": 42
})



In [ ]:
STN_DATA_path_PD = os.path.join(project_root, 'params', 'stn_gpe_params', 'params_PD.yaml')
STN_DATA_path_std_DBS = os.path.join(project_root, 'params', 'stn_gpe_params', 'params_std_DBS.yaml')
STN_DATA_path_sector_DBS = os.path.join(project_root, 'params', 'stn_gpe_params', 'params_sector_DBS.yaml')
DEL_LIM_ARR = [0.1, 0.2, 0.5, 0.8, 1.0] 
FILE_PATHS = [STN_DATA_path_PD, STN_DATA_path_std_DBS, STN_DATA_path_sector_DBS]
TYPE = ['PD', 'STD_DBS', 'SECTOR_DBS']
LAT_STRENGTH = 0.0425

In [ ]:
yaml_path = os.path.join(project_root, 'params', 'decision_making_task_params', 'igt', 'params.yaml')
params = load_yaml(yaml_path)

In [ ]:
TRIALS = params['TRIALS']
EPOCHS = params['EPOCHS']
NUM_BINS = params['NUM_BINS']
LR = params['LR']
NUM_ARMS = params['NUM_ARMS']
SCALING_FACTOR = params['SCALING_FACTOR']
REW_STD = 0
LOSS_STD = 0

In [ ]:
env = IGTEnv(mean_reward=np.array([100,100,50,50])/SCALING_FACTOR,
             std_reward=np.array([REW_STD,REW_STD,REW_STD,REW_STD])/SCALING_FACTOR,
             mean_loss=np.array([-250,-1250,-50,-250])/SCALING_FACTOR,
             std_loss=np.array([LOSS_STD,LOSS_STD,LOSS_STD,LOSS_STD])/SCALING_FACTOR)

In [ ]:
ARM_MONITOR = defaultdict(dict)
NUM_ARM_PICKS_MONITOR = defaultdict(dict)
REWARD_MONITOR = defaultdict(dict)
IGT_SCORE_MONITOR = defaultdict(dict)
IGT_SE_MONITOR = defaultdict(dict)
LAST_BIN_IGT_SCORE_MONITOR = defaultdict(dict)
LAST_BIN_SE_MONITOR = defaultdict(dict)

In [ ]:
t = 0
for file_path in tqdm(FILE_PATHS):
    print(f'Running: {TYPE[t]}')
    args = load_yaml(file_path)
    args['lat_strength_stn'] = LAT_STRENGTH
    save_temp_path = os.path.join(project_root, 'temp', 'temp.yaml')
    save_yaml(args, save_temp_path)
    for del_lim in DEL_LIM_ARR:
        reward_monitor, arm_chosen_monitor, avg_counts,ip_monitor, dp_monitor,ep_monitor, _ = train(env, 
                                                      trails = TRIALS, 
                                                      epochs = EPOCHS, 
                                                      lr = LR, 
                                                      bins = NUM_BINS,
                                                      STN_data = save_temp_path, 
                                                      d1_amp=0.5, #1,
                                                      d2_amp=0.01, #0.0029,
                                                      gpi_threshold=0.15,
                                                      max_gpi_iters=50, 
                                                      del_lim = del_lim,
                                                      del_med = None, printing = False,
                                                      gpi_mean= 1, ep_0 = 0,
                                                      alpha_ep = 0.0,
                                                      eta_ep = 0.1,
                                                      baseline_ep = 0.0, 
                                                      track_arms = False)
        
    
        ARM_MONITOR[TYPE[t]][del_lim] = arm_chosen_monitor
        REWARD_MONITOR[TYPE[t]][del_lim] = reward_monitor

        A_picks = avg_counts[0]
        B_picks = avg_counts[1]
        C_picks = avg_counts[2]
        D_picks = avg_counts[3]

        Avg_A_picks = torch.mean(A_picks, dim = 0)
        Avg_B_picks = torch.mean(B_picks, dim = 0)
        Avg_C_picks = torch.mean(C_picks, dim = 0)
        Avg_D_picks = torch.mean(D_picks, dim = 0)

        IGT_score = torch.mean(torch.add(C_picks, D_picks) - torch.add(A_picks, B_picks), dim = 0).squeeze().numpy()
        IGT_dev = torch.std(torch.add(C_picks, D_picks) - torch.add(A_picks, B_picks), dim = 0)/torch.sqrt(torch.tensor(EPOCHS, dtype = torch.float32))
        IGT_dev = IGT_dev.squeeze().numpy()

        IGT_SCORE_MONITOR[TYPE[t]][del_lim] = IGT_score
        IGT_SE_MONITOR[TYPE[t]][del_lim] = IGT_dev
        LAST_BIN_IGT_SCORE_MONITOR[TYPE[t]][del_lim] = IGT_score[-1]
        LAST_BIN_SE_MONITOR[TYPE[t]][del_lim] = IGT_dev[-1]

        # print(f'IGT score: {IGT_score} with SE of {IGT_dev}')

    
        print(f'Type: {TYPE[t]} and del_lim = {del_lim}: last bin IGT score = {IGT_score[-1]}')
    t +=1 


In [ ]:
DEL_LIM_ARR_FILTERED = [0.1, 0.2, 0.8, 1.0] 

Last_bin_IGT_PD = [LAST_BIN_IGT_SCORE_MONITOR[TYPE[0]][del_lim] for del_lim in DEL_LIM_ARR_FILTERED] 
Last_bin_IGT_std_DBS = [LAST_BIN_IGT_SCORE_MONITOR[TYPE[1]][del_lim] for del_lim in DEL_LIM_ARR_FILTERED]
Last_bin_IGT_sector_DBS = [LAST_BIN_IGT_SCORE_MONITOR[TYPE[2]][del_lim] for del_lim in DEL_LIM_ARR_FILTERED]

Last_bin_IGT_se_PD = [LAST_BIN_SE_MONITOR[TYPE[0]][del_lim] for del_lim in DEL_LIM_ARR_FILTERED] 
Last_bin_IGT_se_std_DBS = [LAST_BIN_SE_MONITOR[TYPE[1]][del_lim] for del_lim in DEL_LIM_ARR_FILTERED]
Last_bin_IGT_se_sector_DBS = [LAST_BIN_SE_MONITOR[TYPE[2]][del_lim] for del_lim in DEL_LIM_ARR_FILTERED]

In [ ]:
# plotting bar plot for each del lim on x axis and prob on y axis. Different colors for each var
bar_width = 0.1
x_values = np.arange(len(DEL_LIM_ARR_FILTERED))
alpha = 1

colors = {'normal':"#054b7c",
          'PD': "#9f0d03ce",
          'std_DBS': "#ad7322",
          'sector_DBS': "#038820",}


#+ np.random.randn(len(Last_bin_Probs_std_DBS))*0.01
plt.figure(figsize=(4,2))
plt.bar(x_values - bar_width, Last_bin_IGT_PD , width=bar_width, label="PD", alpha=alpha, color=colors['PD'], capsize=4)
plt.bar(x_values, Last_bin_IGT_std_DBS, width=bar_width, label="Standard DBS", alpha=alpha, color=colors['std_DBS'], capsize=4)
plt.bar(x_values + bar_width, Last_bin_IGT_sector_DBS, width=bar_width, label="Sector DBS", alpha=alpha, color = colors['sector_DBS'], capsize=4)
plt.legend()
plt.xticks(x_values, [f"{v:.2f}" for v in DEL_LIM_ARR_FILTERED], rotation=0)
plt.xlabel('$\delta_{lim}$')
plt.ylabel('$P(Best arm)$')
# plt.ylim(-0.1, 0.8)
plt.show()

In [ ]:
# saving pickle with lat name
data = {'DEL_LIM_ARR': DEL_LIM_ARR,
        'TYPE': TYPE,
        'ARM_MONITOR': ARM_MONITOR,
        'LAT_STRENGTH': LAT_STRENGTH,
        'LAST_BIN_IGT_SCORE_MONITOR': LAST_BIN_IGT_SCORE_MONITOR,
        'IGT_SCORE_MONITOR': IGT_SCORE_MONITOR,
        'IGT_SE_MONITOR': IGT_SE_MONITOR,
        'LAST_BIN_SE_MONITOR': LAST_BIN_SE_MONITOR,
        'REWARD_MONITOR': REWARD_MONITOR}

file_name = 'igt_data_lat_' + str(LAT_STRENGTH).replace('.', '_') + '.pkl'
file_path = os.path.join(project_root, 'simulations', 'decision_making_tasks', 'igt', file_name)

with open(file_path, 'wb') as f:
    pickle.dump(data, f)

In [40]:
# 